# Warehouse Stock Analysis

## Project Overview
This project analyzes warehouse inventory data to understand stock distribution, pricing patterns, and brand performance.  
The notebook combines exploratory data analysis, descriptive statistics, correlation analysis, categorical association testing, and a Random Forest regression model to explore the main drivers of inventory value.

## Note on data availability
The original dataset is **not included** in this public version of the project.  
To run the notebook locally, place the source file in the `data/` folder and update the path below.

## Main questions
- Which brands and product families account for the largest share of free stock value?
- How is inventory value distributed across product groups?
- Are there meaningful relationships between price, stock, and inventory value?
- Which features are most useful for predicting free stock value?


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Public GitHub version: the original source file is intentionally not included.
# Add your local file to data/warehouse_data.csv before running the notebook.
df = pd.read_csv("data/warehouse_data.csv")


# EDA

In [ ]:
df.shape

In [ ]:
print(f"The table consists of {df.shape[0]} rows and {df.shape[1]} columns.")

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df["COMPANY"].unique()

In [ ]:
df["Warehouse"].unique()

In [ ]:
len( df["Warehouse"].unique())

In [ ]:
df["Product Line"].unique()

In [ ]:
len( df["Product Line"].unique())

In [ ]:
df["Product Family"].unique()

In [ ]:
len( df["Product Family"].unique())

In [ ]:
df["FAMILY"].unique()

In [ ]:
len( df["FAMILY"].unique())

In [ ]:
df["FAMILY"].value_counts().head(30)

In [ ]:
df["BRAND"].unique()

In [ ]:
len( df["BRAND"].unique())

In [ ]:
df["STATUS"].unique()

In [ ]:
len( df["STATUS"].unique())

In [ ]:
df[df.isna().any(axis=1)]


In [ ]:
t1 = df[
    (df['Recommended Price CONSO (BGGE) / CMP (BGEU)'] != 0) &
    (df.notna().all(axis=1))
].drop(columns=['COMPANY', 'REF-COMP-WH'])


In [ ]:
t1.shape

# Descriptive Statistics

In [ ]:
# Total Stock Value (excl. SP and Display Metal Bergner)
total_stock_value = round(t1['Amount in Eur'].sum(), 2)
print(total_stock_value)

In [ ]:
#Average Price per Product in Eur
average_price = round(t1['Recommended Price CONSO (BGGE) / CMP (BGEU)'].mean(),2)
print(average_price)

In [ ]:
#Top Products by Amount in Eur
t1.sort_values(by='Amount in Eur', ascending=False).head(10)

In [ ]:
#Top Products by Recommended Price
t1.sort_values(by='Recommended Price CONSO (BGGE) / CMP (BGEU)', ascending=False).head(10)

In [ ]:
#Top Products by Recommended Price
t1.sort_values(by='Recommended Price CONSO (BGGE) / CMP (BGEU)', ascending=True).head(10)

In [ ]:
#Top Products by Amount free in Euro
t1.sort_values(by='Amount free in Euro', ascending=False).head(10)

In [ ]:
# Brand & Amount Free in Euro
brand_amount_free_df = (
    t1.groupby('BRAND')['Amount free in Euro']
      .sum()
      .sort_values(ascending=False)
      .reset_index()
)
brand_amount_free_df['Amount free in Euro'] = brand_amount_free_df['Amount free in Euro'].apply(lambda x: f"€{x:,.2f}")
print(brand_amount_free_df)


In [ ]:
# Table
brand_amount_free = brand_amount_free_df.style.set_caption("Total Amount of Free Stock in Euro by Brand")                          .set_table_styles([{
                             'selector': 'caption',
                             'props': [('font-size', '16px'), ('font-weight', 'bold')]
                         }]).hide(axis="index")
brand_amount_free


This summary highlights which brands contribute the most to free stock value. A concentrated distribution here suggests that a relatively small number of brands may drive a large share of inventory exposure.

In [ ]:
# Status & Amount Free in Euro
status_amount_free_df = (
    t1.groupby('STATUS')['Amount free in Euro']
      .sum()
      .sort_values(ascending=False)
      .reset_index()
)
status_amount_free_df['Amount free in Euro'] = status_amount_free_df['Amount free in Euro'].apply(lambda x: f"€{x:,.2f}")
print(status_amount_free_df)


In [ ]:
status_amount_free = (
    status_amount_free_df.style
    .set_caption("Total Amount of Free Stock in Euro by Status")
    .set_table_styles([{
        'selector': 'caption',
        'props': [('font-size', '16px'), ('font-weight', 'bold')]
    }]).hide(axis="index")
)
status_amount_free


Comparing status categories helps identify where the highest free stock value sits operationally. This can be useful for prioritizing inventory actions such as liquidation, replenishment, or closer monitoring.

In [ ]:
# Product Count by Brand and Status
brand_status_counts = t1.groupby(['BRAND', 'STATUS']).size().reset_index(name='count')
pivot = brand_status_counts.pivot(index='BRAND', columns='STATUS', values='count').fillna(0).astype(int)
print(pivot)


In [ ]:
brand_status = pivot.style.set_caption("Product Count by Brand and Status") \
    .set_table_styles([{
        'selector': 'caption',
        'props': [('color', 'black'),
                  ('font-size', '16px'),
                  ('font-weight', 'bold'),
                  ('text-align', 'center'),
                  ('padding', '10px')]
    }])

# Display the styled table
brand_status

In [ ]:
# Stock Count by Brand
stock_counts = t1.groupby('BRAND').size().reset_index(name='Stock Count')
stock_counts = stock_counts.sort_values(by='Stock Count', ascending=False)

print(stock_counts)


In [ ]:
#Table
brand_stock_counts = stock_counts.style.set_caption("Stock Count by Brand") \
    .hide(axis="index") \
    .set_table_styles([{
        'selector': 'caption',
        'props': [('color', 'black'),
                  ('font-size', '16px'),
                  ('font-weight', 'bold'),
                  ('text-align', 'center'),
                  ('padding', '10px')]
    }])

brand_stock_counts

In [ ]:
top_30_by_amount = t1.groupby("FAMILY")["Amount free in Euro"].sum().sort_values(ascending=False).head(30)

top_30_df = top_30_by_amount.reset_index()
top_30_df.columns = ['FAMILY', 'Total_Amount_Free']
display(top_30_df)


Looking at the top product families by free stock value helps identify which groups absorb the most capital. These families are likely the most relevant candidates for deeper operational analysis.

# Correlation and Statistical Association

In [ ]:
# Pearson correlation (Numeric vs Numeric)
plt.figure(figsize=(10, 6))
sns.heatmap(t1.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.title("Correlation Matrix of Numeric Variables")
plt.tight_layout()
plt.show()


The heatmap provides a quick view of linear relationships between numeric variables. Stronger coefficients indicate variables that tend to move together, while low correlations suggest a limited linear relationship.

In [ ]:
# Chi-Square Test (Categorical vs Categorical)
from scipy.stats import chi2_contingency

contingency = pd.crosstab(t1['BRAND'], t1['STATUS'])
chi2, p, dof, expected = chi2_contingency(contingency)

print(f"P-value: {p:.6f}")


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
sns.heatmap(contingency, annot=True, fmt='d', cmap='Blues')
plt.title("Product Status Counts per Brand")
plt.ylabel("Brand")
plt.xlabel("Status")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

# Calculate Cramér’s V
n = contingency.sum().sum()
phi2 = chi2 / n
r, k = contingency.shape
cramers_v = np.sqrt(phi2 / min(k - 1, r - 1))

print(f"Cramér’s V: {cramers_v:.3f}")

Cramér’s V complements the Chi-square result by showing the **strength** of the association. This helps distinguish between a statistically significant result and a practically meaningful one.

The Chi-square test checks whether **brand** and **status** are independent. A very small p-value indicates that the distribution of statuses differs meaningfully across brands.

In [ ]:
#| V value   | Strength of Association |
#| --------- | ----------------------- |
#| 0.00–0.10 | Negligible              |
#| 0.10–0.20 | Weak                    |
#| 0.20–0.40 | Moderate                |
#| 0.40–0.60 | Relatively strong       |
#| 0.60–0.80 | Strong                  |
#| 0.80–1.00 | Very strong             |

# Predictive Modeling
Random Forest Regressor

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Prepare modeling dataset
df_model = t1.copy()

# Drop columns that are identifiers, highly descriptive text, or direct leakage
drop_cols = ['Item', 'Description', 'Amount in Eur', 'Product Family']
df_model = df_model.drop(columns=[col for col in drop_cols if col in df_model.columns])

# Keep the 30 most important families by free stock value and group the rest as "Other"
top_30_families = (
    df_model.groupby("FAMILY")["Amount free in Euro"]
    .sum()
    .nlargest(30)
    .index
)
df_model["FAMILY_grouped"] = df_model["FAMILY"].apply(
    lambda x: x if x in top_30_families else "Other"
)
df_model = df_model.drop(columns=["FAMILY"])

# Remove rows with missing target values
df_model = df_model.dropna(subset=["Amount free in Euro"])

# Define features and target
X = df_model.drop(columns=["Amount free in Euro"])
y = df_model["Amount free in Euro"]

# One-hot encode categorical variables
X_encoded = pd.get_dummies(X, drop_first=True)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

# Train model
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

# Predict
y_pred = rf.predict(X_test)

# Evaluate
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)
mae_percent = (mae / y_test.mean()) * 100

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.3f}")
print(f"Mean target value: {y_test.mean():.2f}")
print(f"MAE is about {mae_percent:.1f}% of the average target value")

# Feature importance
feature_importance = (
    pd.Series(rf.feature_importances_, index=X_encoded.columns)
    .sort_values(ascending=False)
    .head(15)
)

plt.figure(figsize=(10, 6))
feature_importance.sort_values().plot(kind="barh")
plt.title("Top 15 Feature Importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

# Prediction error overview
errors_df = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred
})
errors_df["Absolute Error"] = (errors_df["Actual"] - errors_df["Predicted"]).abs()
errors_df.sort_values("Absolute Error", ascending=False).head(10)


The Random Forest model is used here as a flexible baseline for predicting free stock value. The error metrics show how closely the predictions match the observed values, while the feature importance plot helps identify which variables contribute most to the model.

When interpreting the model, **MAE** is often the easiest metric to communicate because it shows the average prediction error in euro terms. **R²** indicates how much of the variation in free stock value is explained by the model.

In [ ]:
# Optional: save the trained model
# import pickle
# with open('random_forest_model.pkl', 'wb') as f:
#     pickle.dump(rf, f)


# Final Insights

- Free stock value appears to be concentrated in a limited set of brands and product families, which suggests that inventory value is not evenly distributed across the catalog.
- Comparing inventory by status provides a clearer operational view of where capital is tied up and where stock management attention may be needed.
- The numeric correlation analysis helps identify which variables move together, while the Chi-square test and Cramér’s V show that categorical structure also matters.
- The Random Forest model demonstrates that inventory value can be predicted reasonably well using available warehouse attributes, with the most important features offering additional business insight.
- A useful next step would be to extend the project with time-based inventory trends, demand patterns, or more detailed error analysis for high-value items.
